In [ ]:
"""
major_json_normalizer.py

Purpose
-------
Normalize Loyola major requirement JSON data so it can be reliably used by the
major recommendation engine.

Why This Is Necessary
---------------------
The catalog scraper produces course requirements in inconsistent formats:

Examples of issues:
    "BIOL\xa0101"                -> non-breaking spaces (invisible mismatch)
    "MATH 131 or MATH 161"      -> inline OR logic
    "BIOL 335/STAT 335"         -> cross-listed courses
    {"options": [...]}           -> already structured choice

The recommender compares student transcripts against major requirements.
If formatting is inconsistent, students appear to have NOT taken courses
they actually completed.

This script standardizes all course requirements into one of two formats:

    Single course:
        "BIOL 101"

    Choice between courses:
        {"options": ["MATH 131", "MATH 161"]}

After running this script:
    • matching becomes deterministic
    • overlap calculations become accurate
    • recommendation quality improves dramatically

This script DOES NOT change academic meaning — only representation.
"""

import json
import re


# ---------------------------------------------------------------------
# Text Normalization Helpers
# ---------------------------------------------------------------------

def clean_spaces(text):
    """
    Replace non-breaking spaces and strip whitespace.

    Loyola catalog HTML uses Unicode \xa0 instead of normal spaces.
    These look identical but break string matching against enrollment data.

    Example:
        "BIOL\xa0101" -> "BIOL 101"

    Args:
        text (str): raw course string

    Returns:
        str: normalized course string
    """
    if isinstance(text, str):
        return text.replace("\xa0", " ").strip()
    return text


def split_or(course):
    """
    Convert inline 'OR' course requirements into structured options.

    Example:
        "MATH 131 or MATH 161"
        ->
        {"options": ["MATH 131", "MATH 161"]}

    The recommender expects structured choices rather than parsing text.

    Args:
        course (str)

    Returns:
        dict | str
    """
    if isinstance(course, str) and " or " in course:
        parts = [c.strip() for c in course.split(" or ")]
        return {"options": parts}
    return course


def split_crosslisted(course):
    """
    Convert cross-listed courses into structured options.

    Example:
        "BIOL 335/STAT 335"
        ->
        {"options": ["BIOL 335", "STAT 335"]}

    Cross-listed courses represent the same class under multiple departments.
    Students only need one — so it must be treated as a choice.

    Args:
        course (str)

    Returns:
        dict | str
    """
    if isinstance(course, str) and "/" in course:
        parts = [c.strip() for c in course.split("/")]
        return {"options": parts}
    return course


def normalize_course(course):
    """
    Apply full normalization pipeline to a course entry.

    Handles:
        • unicode spaces
        • OR logic
        • cross-listed courses
        • existing option objects

    This function guarantees every requirement becomes either:
        "COURSE CODE"
        or
        {"options": [COURSE CODES]}

    Args:
        course (str | dict)

    Returns:
        str | dict
    """

    # Case 1: raw string
    if isinstance(course, str):
        course = clean_spaces(course)
        course = split_crosslisted(course)
        course = split_or(course)
        return course

    # Case 2: already structured options
    if isinstance(course, dict) and "options" in course:
        course["options"] = [clean_spaces(c) for c in course["options"]]
        return course

    return course


# ---------------------------------------------------------------------
# Main Cleaning Routine
# ---------------------------------------------------------------------

def normalize_major_requirements(input_file, output_file):
    """
    Normalize all majors in the scraped catalog dataset.

    Reads the majors JSON, cleans every course requirement, and saves
    a corrected version.

    Args:
        input_file (str): path to raw scraped majors JSON
        output_file (str): path to save cleaned JSON

    Output:
        A structurally identical JSON file with standardized course entries.
    """

    with open(input_file) as f:
        data = json.load(f)

    for program_name, program in data["programs"].items():

        cleaned_courses = []

        for course in program["required_courses"]:
            cleaned_courses.append(normalize_course(course))

        program["required_courses"] = cleaned_courses

    with open(output_file, "w") as f:
        json.dump(data, f, indent=2)

    print(f"Cleaned majors saved to: {output_file}")


# ---------------------------------------------------------------------
# Script Entry Point
# ---------------------------------------------------------------------

if __name__ == "__main__":
    normalize_major_requirements(
        "bachelors_majors_web.json",
        "bachelors_majors_cleaned.json"
    )


In [ ]:
!pwd
!ls


In [ ]:
!git clone https://github.com/thiagopicinini/Stat370-PlanB.git

In [ ]:
!ls

In [ ]:
%cd Stat370-PlanB
!ls

In [ ]:
!find /content -name "*.ipynb"


In [ ]:
%cd /content/Stat370-PlanB


In [ ]:
!git pull


In [ ]:
!git add "data_analysis/EDA Oscar Sanchez Huezca.ipynb"


In [ ]:
!git config --global user.email "kcgandhi04@gmail.com"
!git config --global user.name "Krish Gandhi"


In [ ]:
!git commit -m "Updated EDA and web scraper normalization work"


In [ ]:
from google.colab import auth
auth.authenticate_user()


In [ ]:
import os
os.environ['GITHUB_TOKEN'] = input('Paste your GitHub token here: ')


In [ ]:
!git push https://$GITHUB_TOKEN@github.com/thiagopicinini/Stat370-PlanB.git


In [ ]:
!git status


In [ ]:
!mv -f "/content/Web_scraper_edits.ipynb" "/content/Stat370-PlanB/data_analysis/EDA Oscar Sanchez Huezca.ipynb"


In [ ]:
%cd /content/Stat370-PlanB
!git status


In [ ]:
!git add "data_analysis/EDA Oscar Sanchez Huezca.ipynb"


In [ ]:
!git commit -m "Updated EDA and scraper normalization"


In [ ]:
import os
os.environ['GITHUB_TOKEN'] = input('Paste your NEW GitHub token: ')


In [ ]:
!git push https://$GITHUB_TOKEN@github.com/thiagopicinini/Stat370-PlanB.git
